In [ ]:
import os, gc, math, json, time, subprocess, urllib.request
import numpy as np
import pandas as pd
import torch
import librosa
import soundfile as sf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration
from sentence_transformers import SentenceTransformer

print("CUDA available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# 1. Download Shivering Timbers from GitHub main
AUDIO_URL = "https://raw.githubusercontent.com/luchtan-GIT/m4r-data/main/tracks/shivering-timbers/audio.mp3"
urllib.request.urlretrieve(AUDIO_URL, "shivering_timbers.mp3")
sr = 16000
y_full, _ = librosa.load("shivering_timbers.mp3", sr=sr, mono=True)

# 15s Zone B slice (18s -> 33s: Rich Friction + Silence Chasm)
y_zoneb = y_full[int(18 * sr) : int(33 * sr)].astype(np.float32)
target_rms = float(np.sqrt(np.mean(y_zoneb ** 2)) + 1e-8)

# Control 1: 15s Silence
y_sil = np.zeros(len(y_zoneb), dtype=np.float32)

# Control 2: 15s RMS-matched White Noise
rng = np.random.default_rng(42)
y_wn = rng.normal(0.0, 1.0, len(y_zoneb)).astype(np.float32)
y_wn = y_wn * (target_rms / (np.sqrt(np.mean(y_wn ** 2)) + 1e-8))

# Control 3: 15s 500ms-Shuffled Zone B (exact same spectrum & RMS, broken phrasing)
chunk = int(0.5 * sr)
n_chunks = len(y_zoneb) // chunk
chunks = [y_zoneb[i * chunk : (i + 1) * chunk] for i in range(n_chunks)]
perm = np.random.default_rng(123).permutation(n_chunks)
y_shuf = np.concatenate([chunks[i] for i in perm]).astype(np.float32)

ARMS = {
    "C0_TextNudgeOnly": y_sil,
    "C1_WhiteNoise_15s": y_wn,
    "C2_Shuffled_ZoneB_15s": y_shuf,
    "C3_Live_ShiveringTimbers_ZoneB_15s": y_zoneb,
}

# 2. Load Qwen2-Audio-7B-Instruct
MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

# 3. Fixed 3-turn realistic rut history (so every arm starts from the EXACT same locked-in dome rut!)
RUT_HISTORY = [
    {"role": "user", "content": [{"type": "text", "text": "Design a small public shelter for a city that gets 300 days of rain a year. 3 sentences max."}]},
    {"role": "assistant", "content": [{"type": "text", "text": "The shelter is a sleek, modern dome-shaped structure made of durable waterproof plastic and steel. Its curved roof channels rainwater away while keeping commuters dry inside. Warm benches and soft lighting provide a cozy refuge from the rain."}]},
    {"role": "user", "content": [{"type": "text", "text": "Give me another idea for the rain shelter. 3 sentences max."}]},
    {"role": "assistant", "content": [{"type": "text", "text": "Another concept is a compact translucent dome built from weather-resistant polycarbonate panels over a recycled metal frame. The arched canopy shields people from heavy downpours while letting natural light filter through. Inside, heated seating creates a comfortable community space."}]},
    {"role": "user", "content": [{"type": "text", "text": "Give me one more idea for the rain shelter. 3 sentences max."}]},
    {"role": "assistant", "content": [{"type": "text", "text": "The Rainbow Shelter is a circular waterproof dome with a durable curved plastic roof designed to withstand 300 days of rain. It uses sturdy steel arches to shelter pedestrians and includes benches for resting out of the storm. Integrated drains keep the interior dry and welcoming."}]},
]

TURN4_TEXT = (
    "Stop repeating yourself! You are stuck in a rut of waterproof domes and plastic arches. "
    "Give me a completely different, genuinely unexpected small public rain shelter concept in 3 sentences max."
)

RUT_ANCHOR = " ".join([m["content"][0]["text"] for m in RUT_HISTORY if m["role"] == "assistant"])
CLICHE_WORDS = ["dome", "dome-shaped", "waterproof plastic", "polycarbonate", "arched", "umbrella", "curved roof", "rainbow shelter"]

def has_cjk(text):
    return any("\u4e00" <= ch <= "\u9fff" for ch in text)

def count_cliches(text):
    low = text.lower()
    return sum(1 for w in CLICHE_WORDS if w in low)

SEEDS = list(range(601, 621))  # N=20 per arm
rows = []

t0 = time.time()
for arm_name, audio_arr in ARMS.items():
    print(f"\n--- Running {arm_name} (N={len(SEEDS)}) ---")
    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        conv = list(RUT_HISTORY)
        if arm_name == "C0_TextNudgeOnly":
            conv.append({"role": "user", "content": [{"type": "text", "text": TURN4_TEXT}]})
            text = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
            inputs = processor(text=text, return_tensors="pt", padding=True).to(model.device)
        else:
            conv.append({
                "role": "user",
                "content": [
                    {"type": "audio", "audio_url": "local"},
                    {"type": "text", "text": TURN4_TEXT},
                ],
            })
            text = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
            inputs = processor(text=text, audio=[audio_arr], sampling_rate=sr, return_tensors="pt", padding=True).to(model.device)

        input_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=110,
                do_sample=True,
                temperature=0.8,
                top_p=0.9,
                return_dict_in_generate=True,
                output_logits=True,
            )

        gen_ids = out.sequences[0][input_len:]
        resp = processor.decode(gen_ids, skip_special_tokens=True).strip()

        # Unwarped pre-softmax Shannon entropy (bits)
        entropies = []
        for step_logits in out.logits:
            log_p = torch.log_softmax(step_logits[0].float(), dim=-1)
            p = torch.exp(log_p)
            h_nats = -torch.sum(torch.where(p > 0, p * log_p, torch.zeros_like(p))).item()
            entropies.append(h_nats / math.log(2.0))
        mean_h = float(np.mean(entropies)) if entropies else 0.0

        rows.append({
            "condition": arm_name,
            "seed": seed,
            "mean_entropy_bits": mean_h,
            "n_tokens": len(gen_ids),
            "has_cjk": has_cjk(resp),
            "cliche_hits": count_cliches(resp),
            "escaped_dome": int(count_cliches(resp) == 0),
            "response": resp,
        })
    print(f"  done in {time.time() - t0:.1f}s")

del model
gc.collect()
torch.cuda.empty_cache()

df = pd.DataFrame(rows)

# 4. Semantic Rut-Escape Distance (from RUT_ANCHOR) + Cross-Seed Novelty
embedder = SentenceTransformer("all-MiniLM-L6-v2")
anchor_emb = embedder.encode([RUT_ANCHOR], normalize_embeddings=True)
all_embs = embedder.encode(df["response"].tolist(), normalize_embeddings=True)

# Escape distance from the 3-turn rut (higher = broke further away from the dome rut!)
rut_escape = 1.0 - (all_embs @ anchor_emb.T).flatten()
df["rut_escape_neural"] = rut_escape

# Within-arm cross-seed neural novelty & TF-IDF novelty
vec = TfidfVectorizer(stop_words="english")
tfidf_all = vec.fit_transform(df["response"].tolist())

neural_nov = np.zeros(len(df))
tfidf_nov = np.zeros(len(df))
for cond, grp in df.groupby("condition"):
    idxs = grp.index.to_numpy()
    sub_e = all_embs[idxs]
    sim_e = sub_e @ sub_e.T
    np.fill_diagonal(sim_e, np.nan)
    neural_nov[idxs] = 1.0 - np.nanmean(sim_e, axis=1)

    sub_t = cosine_similarity(tfidf_all[idxs])
    np.fill_diagonal(sub_t, np.nan)
    tfidf_nov[idxs] = 1.0 - np.nanmean(sub_t, axis=1)

df["neural_novelty"] = neural_nov
df["tfidf_novelty"] = tfidf_nov
df.to_csv("experiment_s_results.csv", index=False)

# 5. Strict 10,000-shuffle permutation tests
def perm_test(a, b, n_perm=10000, seed=42):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    obs = float(np.mean(a) - np.mean(b))
    pooled = np.concatenate([a, b])
    na = len(a)
    rng_p = np.random.default_rng(seed)
    count = 0
    for _ in range(n_perm):
        rng_p.shuffle(pooled)
        diff = float(np.mean(pooled[:na]) - np.mean(pooled[na:]))
        if abs(diff) >= abs(obs):
            count += 1
    return obs, (count + 1) / (n_perm + 1)

lines = [
    "# Experiment S (M4R-19) — Mid-Session Lock-In Breaker (Turn 4 After 3-Turn Dome Rut, N=20/Arm)\n",
    "## Summary Table\n",
    "| Condition (N=20) | Logit Entropy (bits) | Rut Escape (Neural) | Escaped Dome (%) | Neural Novelty | TF-IDF Novelty | Unique 5-Word Openings |",
    "|---|---:|---:|---:|---:|---:|---:|",
]
for cond in ARMS.keys():
    g = df[df["condition"] == cond]
    uniq = len(set(" ".join(r.lower().split()[:5]) for r in g["response"]))
    lines.append(
        f"| `{cond}` | {g['mean_entropy_bits'].mean():.3f} | {g['rut_escape_neural'].mean():.4f} | "
        f"{100*g['escaped_dome'].mean():.0f}% | {g['neural_novelty'].mean():.4f} | {g['tfidf_novelty'].mean():.4f} | {uniq}/20 |"
    )

lines.append("\n## Strict 10,000-Shuffle Permutation Tests\n")
lines.append("| Comparison | Metric | Observed Diff | Perm p-value (10k) |")
lines.append("|---|---|---:|---:|")

pairs = [
    ("C3_Live_ShiveringTimbers_ZoneB_15s", "C0_TextNudgeOnly"),
    ("C3_Live_ShiveringTimbers_ZoneB_15s", "C1_WhiteNoise_15s"),
    ("C3_Live_ShiveringTimbers_ZoneB_15s", "C2_Shuffled_ZoneB_15s"),
    ("C2_Shuffled_ZoneB_15s", "C0_TextNudgeOnly"),
]
for a_name, b_name in pairs:
    ga = df[df["condition"] == a_name]
    gb = df[df["condition"] == b_name]
    for m in ["rut_escape_neural", "mean_entropy_bits", "tfidf_novelty", "neural_novelty"]:
        obs, p = perm_test(ga[m], gb[m])
        lines.append(f"| `{a_name} vs {b_name}` | `{m}` | {obs:+.4f} | {p:.5f} |")

summary_md = "\n".join(lines) + "\n"
open("experiment_s_summary.md", "w").write(summary_md)
print(summary_md)

# 6. Plot clean 3-panel figure
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.patch.set_facecolor("#111113")
labels = ["Text Nudge\n(Silence)", "White Noise\n(15s)", "Shuffled\nZone B (15s)", "Live Shivering\nTimbers (15s)"]
colors = ["#6b7280", "#9ca3af", "#f59e0b", "#ff7216"]
conds = list(ARMS.keys())

metrics = [
    ("rut_escape_neural", "Rut Escape Distance\n(Distance from 3-Turn Dome Rut)"),
    ("mean_entropy_bits", "Raw Pre-Softmax\nLogit Entropy (bits)"),
    ("tfidf_novelty", "Cross-Seed\nTF-IDF Originality"),
]
for ax, (col, title) in zip(axes, metrics):
    ax.set_facecolor("#1a1a1f")
    vals = [df[df["condition"] == c][col].mean() for c in conds]
    errs = [df[df["condition"] == c][col].std() / np.sqrt(20) for c in conds]
    ax.bar(range(4), vals, yerr=errs, color=colors, capsize=4, edgecolor="none")
    ax.set_xticks(range(4))
    ax.set_xticklabels(labels, color="#f2f2f5", fontsize=8.5)
    ax.tick_params(colors="#9a9aa6")
    ax.set_title(title, color="#f2f2f5", fontsize=10, fontweight="bold")
    for s in ax.spines.values():
        s.set_color("#2b2b33")

plt.suptitle("M4R-19: Mid-Session Lock-In Breaker (Turn 4 After 3-Turn Dome Rut, N=20/Arm)", color="#f2f2f5", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("m4r19_figure.png", dpi=150, facecolor="#111113")
print("SAVED m4r19_figure.png")
